In [45]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve,
    roc_auc_score
)

---

# Load all metrics for all models in a dictionary

In [46]:
load_model_bool = False
load_metrics_bool = True


parent_folder = "classification_ring/models"

models_dict = {
    "logistic": parent_folder + "/logistic_classifier",
    "random_forest": parent_folder + "/random_forest",
}


if load_model_bool:
    models = {}
    import pickle
    for model_name, model_folder in models_dict.items():
        model_path = f"{model_folder}/{model_name}_model.pkl"
        with open(model_path, "rb") as f:
            models[model_name] = pickle.load(f)
        print(f"Model loaded from {model_path}")


if load_metrics_bool:
    metrics_dict = {}
    for model_name, model_folder in models_dict.items():
        # Postprocessed results paths
        postprocessed_threshold_results_path = f"{model_folder}/postprocessed_threshold_results.parquet"
        postprocessed_curve_results_path = f"{model_folder}/postprocessed_curve_results.parquet"
        # Original results paths
        threshold_results_path = f"{model_folder}/threshold_results.parquet"
        curve_results_path = f"{model_folder}/curve_results.parquet"

        metrics_dict[model_name] = {
            "postprocessed_threshold_results": pd.read_parquet(postprocessed_threshold_results_path),
            "postprocessed_curve_results": pd.read_parquet(postprocessed_curve_results_path),
            "threshold_results": pd.read_parquet(threshold_results_path),
            "curve_results": pd.read_parquet(curve_results_path)
        }
        print(f"Metrics loaded for {model_name} from {model_folder}")

Metrics loaded for logistic from classification_ring/models/logistic_classifier
Metrics loaded for random_forest from classification_ring/models/random_forest


In [47]:
def summarize_existing_threshold_df(threshold_df, fixed_threshold=0.5):

    rows = []

    for class_name, class_df in threshold_df.groupby("class"):

        fixed_row = class_df[
            np.isclose(class_df["threshold"], fixed_threshold)
        ].iloc[0]

        best_row = (
            class_df
            .sort_values("MCC", ascending=False)
            .iloc[0]
        )

        rows.append({
            "class": class_name,
            "MCC_at_0_5": fixed_row["MCC"],
            "BA_at_0_5": fixed_row["balanced_accuracy"],
            "best_MCC": best_row["MCC"],
            "best_threshold": best_row["threshold"],
            "BA_at_best_MCC": best_row["balanced_accuracy"]
        })

    return pd.DataFrame(rows)






def compare_models(metrics_dict):
    results_comparison_dict = {}

    label_cols = [
        "HBOND",
        "VDW",
        "IONIC",
        "PIPISTACK",
        "PICATION",
        "SSBOND",
        "PIHBOND"
    ]
    fixed_threshold = 0.5

    for model_name, model_metrics in metrics_dict.items():

        print(f"Processing metrics for model: {model_name}")

        baseline_threshold_results_df = model_metrics["threshold_results"]
        postprocessed_threshold_results_df = model_metrics["postprocessed_threshold_results"]
        baseline_curve_results_df = model_metrics["curve_results"]
        postprocessed_curve_results_df = model_metrics["postprocessed_curve_results"]

        # ------------------------------------------------------------
        # 2. Curve summaries: ROC-AUC, AP
        # ------------------------------------------------------------

        baseline_curve_summary_df = (
            baseline_curve_results_df[
                ["class", "average_precision", "ROC_AUC"]
            ]
        )

        postprocessed_curve_summary_df = (
            postprocessed_curve_results_df[
                ["class", "average_precision", "ROC_AUC"]
            ]
        )


        # ------------------------------------------------------------
        # 3. Threshold summaries from existing threshold grids
        #    - fixed MCC@0.5
        #    - best MCC and threshold
        # ------------------------------------------------------------


        baseline_threshold_summary_df = summarize_existing_threshold_df(
            baseline_threshold_results_df,
            fixed_threshold=fixed_threshold
        )

        postprocessed_threshold_summary_df = summarize_existing_threshold_df(
            postprocessed_threshold_results_df,
            fixed_threshold=fixed_threshold
        )


        # ------------------------------------------------------------
        # 4. Merge everything
        # ------------------------------------------------------------

        metric_comparison_df = (
            baseline_curve_summary_df
            .merge(
                postprocessed_curve_summary_df,
                on="class",
                suffixes=("_baseline", "_postprocessed")
            )
            .merge(
                baseline_threshold_summary_df,
                on="class"
            )
            .rename(columns={
                "MCC_at_0_5": "MCC_at_0_5_baseline",
                "BA_at_0_5": "BA_at_0_5_baseline",
                "best_MCC": "best_MCC_baseline",
                "best_threshold": "best_threshold_baseline",
                "BA_at_best_MCC": "BA_at_best_MCC_baseline"
            })
            .merge(
                postprocessed_threshold_summary_df,
                on="class"
            )
            .rename(columns={
                "MCC_at_0_5": "MCC_at_0_5_postprocessed",
                "BA_at_0_5": "BA_at_0_5_postprocessed",
                "best_MCC": "best_MCC_postprocessed",
                "best_threshold": "best_threshold_postprocessed",
                "BA_at_best_MCC": "BA_at_best_MCC_postprocessed"
            })
        )


        # ------------------------------------------------------------
        # 5. Add deltas
        # ------------------------------------------------------------

        for metric in [
            "ROC_AUC",
            "average_precision",
            "MCC_at_0_5",
            "BA_at_0_5",
            "best_MCC",
            "BA_at_best_MCC"
        ]:

            metric_comparison_df[f"delta_{metric}"] = (
                metric_comparison_df[f"{metric}_postprocessed"]
                - metric_comparison_df[f"{metric}_baseline"]
            )


        # ------------------------------------------------------------
        # 6. Preserve original label order
        # ------------------------------------------------------------

        metric_comparison_df["class"] = pd.Categorical(
            metric_comparison_df["class"],
            categories=label_cols,
            ordered=True
        )

        metric_comparison_df = (
            metric_comparison_df
            .sort_values("class")
            .reset_index(drop=True)
        )

        # ------------------------------------------------------------
        # 7. Store the final comparison dataframe for this model
        # ------------------------------------------------------------
        results_comparison_dict[model_name] = metric_comparison_df
    return results_comparison_dict

In [48]:
results_comparison_dict = compare_models(metrics_dict)

Processing metrics for model: logistic
Processing metrics for model: random_forest


In [49]:
delta_cols = [col for col in results_comparison_dict["logistic"].columns if col.startswith("delta_")]

results_comparison_dict["logistic"][["class"] + delta_cols]

,class,delta_ROC_AUC,delta_average_precision,delta_MCC_at_0_5,delta_BA_at_0_5,delta_best_MCC,delta_BA_at_best_MCC
0,HBOND,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
1,VDW,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
2,IONIC,0.017882,1.836556e-01,0.153580,0.028323,0.147244,0.034257
3,PIPISTACK,0.014428,3.170207e-01,0.284835,0.019944,0.229188,0.021159
4,PICATION,0.021572,2.037421e-01,0.137669,0.021548,0.131872,0.022834
5,SSBOND,0.000000,-1.110223e-16,0.008191,0.000023,0.000000,0.000000
6,PIHBOND,0.018008,7.953368e-04,0.007226,0.015717,0.005356,0.145287


In [50]:
results_comparison_dict["random_forest"][["class"] + delta_cols]

,class,delta_ROC_AUC,delta_average_precision,delta_MCC_at_0_5,delta_BA_at_0_5,delta_best_MCC,delta_BA_at_best_MCC
0,HBOND,0.000000e+00,0.000000e+00,0.0,0.0,0.000000,0.000000
1,VDW,0.000000e+00,0.000000e+00,0.0,0.0,0.000000,0.000000
2,IONIC,1.497963e-05,5.779259e-05,0.0,0.0,0.000000,0.000000
3,PIPISTACK,-1.110223e-16,-1.110223e-16,0.0,0.0,0.000000,0.000000
4,PICATION,3.259924e-05,2.060812e-04,0.0,0.0,0.000163,0.000008
5,SSBOND,0.000000e+00,0.000000e+00,0.0,0.0,0.000000,0.000000
6,PIHBOND,1.052621e-03,1.330435e-04,0.0,0.0,0.000000,0.000000


---
# All models, key metrics compared

In [51]:
def make_postprocessed_model_comparison_table(results_comparison_dict, model_name = "logistic"):

    rows = []

    for model_name, df in results_comparison_dict.items():

        tmp = df[[
            "class",
            "average_precision_postprocessed",
            "ROC_AUC_postprocessed",
            "MCC_at_0_5_postprocessed",
            "BA_at_0_5_postprocessed",
            "best_MCC_postprocessed",
            "best_threshold_postprocessed"
        ]].copy()

        tmp.insert(0, "model", model_name)

        tmp = tmp.rename(columns={
            "average_precision_postprocessed": "AP",
            "ROC_AUC_postprocessed": "ROC-AUC",
            "MCC_at_0_5_postprocessed": "MCC@0.5",
            "BA_at_0_5_postprocessed": "BA@0.5",
            "best_MCC_postprocessed": "best MCC",
            "best_threshold_postprocessed": "best threshold"
        })

        rows.append(tmp)

    comparison_df = pd.concat(rows, ignore_index=True)

    label_order = [
        "HBOND",
        "VDW",
        "IONIC",
        "PIPISTACK",
        "PICATION",
        "SSBOND",
        "PIHBOND"
    ]

    comparison_df["class"] = pd.Categorical(
        comparison_df["class"],
        categories=label_order,
        ordered=True
    )

    comparison_df = (
        comparison_df
        .sort_values(["class", "model"])
        .reset_index(drop=True)
    )

    return comparison_df

In [52]:
postprocessed_model_comparison_df = make_postprocessed_model_comparison_table(
    results_comparison_dict
)

postprocessed_model_comparison_df

,model,class,AP,ROC-AUC,MCC@0.5,BA@0.5,best MCC,best threshold
0,logistic,HBOND,0.827871,0.668310,0.218933,0.621051,0.228122,0.40
1,random_forest,HBOND,0.888615,0.773537,0.365905,0.637097,0.383716,0.63
2,logistic,VDW,0.586224,0.592472,0.129391,0.564682,0.130832,0.49
3,random_forest,VDW,0.650049,0.646743,0.196914,0.597658,0.199175,0.55
4,logistic,IONIC,0.499206,0.983874,0.553536,0.969451,0.566812,0.70
5,random_forest,IONIC,0.573156,0.986119,0.450495,0.664452,0.582207,0.19
6,logistic,PIPISTACK,0.775792,0.995889,0.852711,0.995439,0.852766,0.66
7,random_forest,PIPISTACK,0.846539,0.997097,0.842886,0.965882,0.854529,0.24
8,logistic,PICATION,0.287473,0.990329,0.407854,0.983506,0.442261,0.73
9,random_forest,PICATION,0.363938,0.989939,0.205409,0.541119,0.470652,0.05


In [53]:
report_postprocessed_model_comparison_df = postprocessed_model_comparison_df[[
    "class",
    "model",
    "AP",
    "ROC-AUC",
    #"MCC@0.5",
    "best MCC",
    "best threshold"
]].copy()

numeric_cols = report_postprocessed_model_comparison_df.select_dtypes(
    include="number"
).columns

report_postprocessed_model_comparison_df[numeric_cols] = (
    report_postprocessed_model_comparison_df[numeric_cols]
    .round(3)
)

report_postprocessed_model_comparison_df

,class,model,AP,ROC-AUC,best MCC,best threshold
0,HBOND,logistic,0.828,0.668,0.228,0.40
1,HBOND,random_forest,0.889,0.774,0.384,0.63
2,VDW,logistic,0.586,0.592,0.131,0.49
3,VDW,random_forest,0.650,0.647,0.199,0.55
4,IONIC,logistic,0.499,0.984,0.567,0.70
5,IONIC,random_forest,0.573,0.986,0.582,0.19
6,PIPISTACK,logistic,0.776,0.996,0.853,0.66
7,PIPISTACK,random_forest,0.847,0.997,0.855,0.24
8,PICATION,logistic,0.287,0.990,0.442,0.73
9,PICATION,random_forest,0.364,0.990,0.471,0.05


In [54]:
latex_table = report_postprocessed_model_comparison_df.rename(columns={
    "class": "Class",
    "model": "Model",
    "best MCC": "Best MCC",
    "AP": "AP",
    "ROC-AUC": "ROC-AUC",
    "best threshold": "Best threshold"
}).to_latex(
    index=False,
    escape=False,
    column_format="llccccc",
    float_format="%.3f"
)

latex_table = latex_table.replace("\\begin{tabular}", "\\begin{tabular}")
print(latex_table)

\begin{tabular}{llccccc}
\toprule
Class & Model & AP & ROC-AUC & Best MCC & Best threshold \\
\midrule
HBOND & logistic & 0.828 & 0.668 & 0.228 & 0.400 \\
HBOND & random_forest & 0.889 & 0.774 & 0.384 & 0.630 \\
VDW & logistic & 0.586 & 0.592 & 0.131 & 0.490 \\
VDW & random_forest & 0.650 & 0.647 & 0.199 & 0.550 \\
IONIC & logistic & 0.499 & 0.984 & 0.567 & 0.700 \\
IONIC & random_forest & 0.573 & 0.986 & 0.582 & 0.190 \\
PIPISTACK & logistic & 0.776 & 0.996 & 0.853 & 0.660 \\
PIPISTACK & random_forest & 0.847 & 0.997 & 0.855 & 0.240 \\
PICATION & logistic & 0.287 & 0.990 & 0.442 & 0.730 \\
PICATION & random_forest & 0.364 & 0.990 & 0.471 & 0.050 \\
SSBOND & logistic & 0.868 & 1.000 & 0.867 & 0.000 \\
SSBOND & random_forest & 0.923 & 1.000 & 0.908 & 0.420 \\
PIHBOND & logistic & 0.008 & 0.928 & 0.064 & 0.420 \\
PIHBOND & random_forest & 0.034 & 0.761 & 0.106 & 0.250 \\
\bottomrule
\end{tabular}



### Single model, baseline vs postprocessed

In [56]:
def single_model_baseline_vs_postprocessed_comparison_df(metrics_dict, model_name):

    baseline_threshold_results_df = metrics_dict[model_name]["threshold_results"]
    postprocessed_threshold_results_df = metrics_dict[model_name]["postprocessed_threshold_results"]
    baseline_curve_results_df = metrics_dict[model_name]["curve_results"]
    postprocessed_curve_results_df = metrics_dict[model_name]["postprocessed_curve_results"]
    
    label_cols = baseline_curve_results_df["class"].unique().tolist()
    fixed_threshold = 0.5


    # ------------------------------------------------------------
    # 2. Curve summaries: ROC-AUC, AP, AP/prevalence
    # ------------------------------------------------------------
    baseline_curve_summary_df = (
        baseline_curve_results_df[
            ["class", "average_precision", "ROC_AUC"]
        ]
    )

    postprocessed_curve_summary_df = (
        postprocessed_curve_results_df[
            ["class", "average_precision", "ROC_AUC"]
        ]
    )

    # ------------------------------------------------------------
    # 3. Threshold summaries from existing threshold grids
    #    - fixed MCC@0.5
    #    - best MCC and threshold
    # ------------------------------------------------------------

    def summarize_existing_threshold_df(threshold_df, fixed_threshold=0.5):

        rows = []

        for class_name, class_df in threshold_df.groupby("class"):

            fixed_row = class_df[
                np.isclose(class_df["threshold"], fixed_threshold)
            ].iloc[0]

            best_row = (
                class_df
                .sort_values("MCC", ascending=False)
                .iloc[0]
            )

            rows.append({
                "class": class_name,
                "MCC_at_0_5": fixed_row["MCC"],
                "BA_at_0_5": fixed_row["balanced_accuracy"],
                "best_MCC": best_row["MCC"],
                "best_threshold": best_row["threshold"],
                "BA_at_best_MCC": best_row["balanced_accuracy"]
            })

        return pd.DataFrame(rows)


    baseline_threshold_summary_df = summarize_existing_threshold_df(
        baseline_threshold_results_df,
        fixed_threshold=fixed_threshold
    )

    postprocessed_threshold_summary_df = summarize_existing_threshold_df(
        postprocessed_threshold_results_df,
        fixed_threshold=fixed_threshold
    )


    # ------------------------------------------------------------
    # 4. Merge everything
    # ------------------------------------------------------------

    metric_comparison_df = (
        baseline_curve_summary_df
        .merge(
            postprocessed_curve_summary_df,
            on="class",
            suffixes=("_baseline", "_postprocessed")
        )
        .merge(
            baseline_threshold_summary_df,
            on="class"
        )
        .rename(columns={
            "MCC_at_0_5": "MCC_at_0_5_baseline",
            "BA_at_0_5": "BA_at_0_5_baseline",
            "best_MCC": "best_MCC_baseline",
            "best_threshold": "best_threshold_baseline",
            "BA_at_best_MCC": "BA_at_best_MCC_baseline"
        })
        .merge(
            postprocessed_threshold_summary_df,
            on="class"
        )
        .rename(columns={
            "MCC_at_0_5": "MCC_at_0_5_postprocessed",
            "BA_at_0_5": "BA_at_0_5_postprocessed",
            "best_MCC": "best_MCC_postprocessed",
            "best_threshold": "best_threshold_postprocessed",
            "BA_at_best_MCC": "BA_at_best_MCC_postprocessed"
        })
    )


    # ------------------------------------------------------------
    # 5. Add deltas
    # ------------------------------------------------------------

    for metric in [
        "ROC_AUC",
        "average_precision",
        "MCC_at_0_5",
        "BA_at_0_5",
        "best_MCC",
        "BA_at_best_MCC"
    ]:

        metric_comparison_df[f"delta_{metric}"] = (
            metric_comparison_df[f"{metric}_postprocessed"]
            - metric_comparison_df[f"{metric}_baseline"]
        )


    # ------------------------------------------------------------
    # 6. Preserve original label order
    # ------------------------------------------------------------


    metric_comparison_df["class"] = pd.Categorical(
        metric_comparison_df["class"],
        categories=label_cols,
        ordered=True
    )

    metric_comparison_df = (
        metric_comparison_df
        .sort_values("class")
        .reset_index(drop=True)
    )

    return metric_comparison_df

In [57]:
baseline_vs_postprocessed_comparison_df = single_model_baseline_vs_postprocessed_comparison_df(
    metrics_dict,
    model_name="logistic"
)
baseline_vs_postprocessed_comparison_df

,class,average_precision_baseline,ROC_AUC_baseline,average_precision_postprocessed,ROC_AUC_postprocessed,MCC_at_0_5_baseline,BA_at_0_5_baseline,best_MCC_baseline,best_threshold_baseline,BA_at_best_MCC_baseline,...,BA_at_0_5_postprocessed,best_MCC_postprocessed,best_threshold_postprocessed,BA_at_best_MCC_postprocessed,delta_ROC_AUC,delta_average_precision,delta_MCC_at_0_5,delta_BA_at_0_5,delta_best_MCC,delta_BA_at_best_MCC
0,HBOND,0.827871,0.668310,0.827871,0.668310,0.218933,0.621051,0.228122,0.40,0.604821,...,0.621051,0.228122,0.40,0.604821,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
1,VDW,0.586224,0.592472,0.586224,0.592472,0.129391,0.564682,0.130832,0.49,0.565240,...,0.564682,0.130832,0.49,0.565240,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
2,IONIC,0.315551,0.965992,0.499206,0.983874,0.399956,0.941128,0.419568,0.74,0.912452,...,0.969451,0.566812,0.70,0.946710,0.017882,1.836556e-01,0.153580,0.028323,0.147244,0.034257
3,PIPISTACK,0.458772,0.981461,0.775792,0.995889,0.567876,0.975495,0.623578,0.82,0.974282,...,0.995439,0.852766,0.66,0.995441,0.014428,3.170207e-01,0.284835,0.019944,0.229188,0.021159
4,PICATION,0.083731,0.968756,0.287473,0.990329,0.270185,0.961959,0.310388,0.77,0.939048,...,0.983506,0.442261,0.73,0.961883,0.021572,2.037421e-01,0.137669,0.021548,0.131872,0.022834
5,SSBOND,0.868147,0.999827,0.868147,0.999827,0.858327,0.999677,0.866518,0.98,0.999700,...,0.999700,0.866518,0.00,0.999700,0.000000,-1.110223e-16,0.008191,0.000023,0.000000,0.000000
6,PIHBOND,0.007065,0.909787,0.007860,0.927795,0.053326,0.838450,0.058158,0.81,0.744815,...,0.854167,0.063515,0.42,0.890102,0.018008,7.953368e-04,0.007226,0.015717,0.005356,0.145287
